Here's the training code I used. It's pretty standard stuff, nothing more complex than what you'd find in open-source codes. So it's just for an example.

ref:
- https://www.kaggle.com/code/takamichitoda/dpc-starter-train

In [2]:
%%writefile run.sh
#!/bin/bash

# 定义一个运行实验的函数
run_experiment() {
    local TRAIN_DS=$1
    local VALID_DS=$2
    local RUN_NAME=$3
    local RUN_DIR="./output/${RUN_NAME}"

    echo "================================================================="
    echo "🚀 开始实验: $RUN_NAME"
    echo "   TRAIN_DS: $TRAIN_DS"
    echo "   VALID_DS: $VALID_DS"
    echo "   RUN_DIR:  $RUN_DIR"
    echo "================================================================="

    # 创建文件夹（如果不存在）
    mkdir -p "$RUN_DIR"

    # 启动程序
    accelerate launch --config_file acc_config.yaml train-byt5.py \
        --TRAIN_DS_PATH "$TRAIN_DS" \
        --VALID_DS_PATH "$VALID_DS" \
        --OUTPUT_DIR "$RUN_DIR" \
        --RUN_NAME "$RUN_NAME" 2>&1 | tee "$RUN_DIR/my_log.txt"

    echo "✅ 实验 $RUN_NAME 已完成！日志路径: $RUN_DIR/my_log.txt"
    echo ""
}

run_experiment \
    "/2022533109/guguoqin/deep-past/datasets/data1/all.json" \
    "None" \
    "byt5-xl/hf-20260323-data1-all"

run_experiment \
    "/2022533109/guguoqin/deep-past/datasets/data1_llmlabel/all.json" \
    "None" \
    "byt5-xl/hf-20260323-data1_llmlabel-all"

run_experiment \
    "/2022533109/guguoqin/deep-past/datasets/data2_llmlabel_synth12/all.json" \
    "None" \
    "byt5-xl/hf-20260323-data2_llmlabel_synth12-all"

run_experiment \
    "/2022533109/guguoqin/deep-past/datasets/data3_llmlabel_synth12/all.json" \
    "None" \
    "byt5-xl/hf-20260323-data3_llmlabel_synth12-all"

echo "🎉 所有实验运行完毕！"

Writing run.sh


In [3]:
%%writefile train-byt5.py
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--TRAIN_DS_PATH", type=str, default="/2022533109/guguoqin/deep-past/datasets/data2_llmlabel_synth2/all.json")
parser.add_argument("--VALID_DS_PATH", type=str, default=None)
parser.add_argument("--OUTPUT_DIR", type=str, default="./output/byt5-xl/hf-20260321-data2_llmlabel_synth2-all")
parser.add_argument("--RUN_NAME", type=str, default="byt5-xl/hf-20260321-data2_llmlabel_synth2-all")
args = parser.parse_args()
if args.VALID_DS_PATH == "None":
    args.VALID_DS_PATH = None

class Config:
    # Train
    PREFIX_EN = "Translate Akkadian to English: "
    PREFIX_TR = "Translate Akkadian to Turkish: "
    PREFIX_DE = "Translate Akkadian to German: "
    TRAIN_MAX_LENGTH = 1024
    
    EPOCHS = 3
    LEARNING_RATE = 2e-4
    MIN_LEARNING_RATE = 1.33e-4 # 0.66e-4
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 1

    # Basic settings
    CUDA_VISIBLE_DEVICES = "0,1,2,3,4,5,6,7"
    MODEL_NAME = "/2022533109/guguoqin/models/google/byt5-xl"

    TRAIN_DS_PATH = args.TRAIN_DS_PATH
    VALID_DS_PATH = args.VALID_DS_PATH

    OUTPUT_DIR = args.OUTPUT_DIR
    RUN_NAME = args.RUN_NAME

    # Inference
    DO_INFER = False
    INFER_MAX_LENGTH = 1024
    INFER_NUM_BEAMS = 4
    INFER_MAX_NEW_TOKENS = 512
    INFER_BATCH_SIZE = 16
    TARGET_CKPT = 3

import os
os.environ["CUDA_VISIBLE_DEVICES"] = Config.CUDA_VISIBLE_DEVICES
os.environ["SWANLAB_PROJECT"]="deep-past"
os.environ["SWANLAB_LOG_DIR"]=Config.OUTPUT_DIR
import gc
import re
import math
import json
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
import evaluate

def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
seed_everything()

# ==========================================
# 1. Load data
# ==========================================
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = []
    for item in data:
        df.append({
            "uuid": item["uuid"],
            "publication_name": item['publication_name'],
            "chapter_name": item['chapter_name'],
            "sentence_id": item.get("sentence_id", 1),
            "target_lang_code": item["messages"][0]["content"][0]["target_lang_code"],
            "transliteration": item["messages"][0]["content"][0]["text"],
            "translation": item["messages"][1]["content"]
        })
    df = pd.DataFrame(df)
    return df

def get_token_limits(transliteration_len):
    # 根据你的拟合结果填入系数
    m, c = 1.1427, -1.1173
    up, down = 200, 200
    
    max_tokens = int(m * transliteration_len + (c + up))
    min_tokens = max(0, int(m * transliteration_len + (c - down)))
    
    return min_tokens, max_tokens

train_df = load_dataset(Config.TRAIN_DS_PATH)
print(f"Load Train Data: {Config.TRAIN_DS_PATH}")
train_df['transliteration_char_length'] = train_df.apply(lambda row: len(row['transliteration']), axis=1)
train_df['translation_char_length'] = train_df.apply(lambda row: len(row['translation']), axis=1)
print(f"Original Train Data: {len(train_df)} docs")
print(f"Desc of transliteration_char_length: {train_df['transliteration_char_length'].describe()}")
print(f"Desc of translation_char_length: {train_df['translation_char_length'].describe()}")
if Config.VALID_DS_PATH:
    valid_df = load_dataset(Config.VALID_DS_PATH)
    valid_df['transliteration_char_length'] = valid_df.apply(lambda row: len(row['transliteration']), axis=1)
    valid_df['translation_char_length'] = valid_df.apply(lambda row: len(row['translation']), axis=1)
    valid_df = valid_df.sort_values(by='transliteration_char_length', ascending=False).reset_index(drop=True)
    valid_df[['min_t', 'max_t']] = valid_df.apply(
        lambda row: get_token_limits(row['transliteration_char_length']), 
        axis=1, result_type='expand'
    )
    print(f"Original Valid Data: {len(valid_df)} docs")
    print(f"Desc of transliteration_char_length: {valid_df['transliteration_char_length'].describe()}")
    print(f"Desc of translation_char_length: {valid_df['translation_char_length'].describe()}")

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df)
if Config.VALID_DS_PATH:
    valid_dataset = Dataset.from_pandas(valid_df)
    effective_batch_size = Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS
    steps_per_epoch = math.ceil(len(train_dataset) / effective_batch_size)
    target_step = steps_per_epoch * Config.TARGET_CKPT
    dynamic_ckpt_path = os.path.join(Config.OUTPUT_DIR, f"checkpoint-{target_step}")
    print(f"Calculated steps per epoch: {steps_per_epoch}")
    print(f"Targeting INFER CKPT {Config.TARGET_CKPT} -> Will loading checkpoint from: {dynamic_ckpt_path}")

# ==========================================
# 2. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

def preprocess_function(examples):
    inputs = []
    for lang_code, ex in zip(examples["target_lang_code"], examples["transliteration"]):
        if lang_code == "en":
            inputs.append(Config.PREFIX_EN + str(ex))
        elif lang_code == "tr":
            inputs.append(Config.PREFIX_TR + str(ex))
        elif lang_code == "de":
            inputs.append(Config.PREFIX_DE + str(ex))
    targets = [str(ex) for ex in examples["translation"]]
    
    model_inputs = tokenizer(inputs, max_length=Config.TRAIN_MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=Config.TRAIN_MAX_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
if Config.VALID_DS_PATH:
    tokenized_valid = valid_dataset.map(preprocess_function, batched=True)

# ==========================================
# 3. Model training
# ==========================================
gc.collect()
torch.cuda.empty_cache()
model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(
    tokenizer, 
    model=model
)

# Metric
chrf_metric = evaluate.load("evaluate/metrics/chrf/chrf.py")
sacrebleu_metric = evaluate.load("evaluate/metrics/sacrebleu/sacrebleu.py")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): 
        preds = preds[0]
    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)
    
    preds = preds.astype(np.int64)
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    chrf_result = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)
    chrf_score = chrf_result["score"]
    
    bleu_result = sacrebleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    bleu_score = bleu_result["score"]
    
    geo_mean = (chrf_score * bleu_score) ** 0.5 if chrf_score > 0 and bleu_score > 0 else 0.0
    
    return {
        "chrf": chrf_score,
        "bleu": bleu_score,
        "geo_mean": geo_mean
    }

class FocalLossCallback:
    def __init__(self, gamma=1.0, alpha=1.0, ignore_index=-100):
        """
        初始化 Focal Loss
        :param gamma: 聚焦参数，越大越关注难分类的样本 (通常取 2.0)
        :param alpha: 平衡参数 (通常取 1.0，除非你有明确的类别权重需求)
        :param ignore_index: 忽略的 Label 索引 (HuggingFace 默认为 -100)
        """
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def __call__(self, outputs, labels, num_items_in_batch=None):
        """
        符合 HF Trainer compute_loss_func 签名的调用函数
        """
        # 1. 维度展平 (Flatten)
        # Logits: [batch_size, seq_len, vocab_size] -> [N, vocab_size]
        # Labels: [batch_size, seq_len]             -> [N]
        logits = outputs.get("logits") if isinstance(outputs, dict) else outputs[0]
        logits = logits.view(-1, logits.size(-1))
        labels = labels.view(-1)

        # 2. 计算基础 Cross Entropy (不进行 reduce，保留每个 Token 的 loss)
        ce_loss = F.cross_entropy(
            logits, 
            labels, 
            reduction='none', 
            ignore_index=self.ignore_index
        )

        # 3. 计算概率 pt
        # ce_loss = -log(pt) => pt = exp(-ce_loss)
        pt = torch.exp(-ce_loss)

        # 4. 计算 Focal Loss: - alpha * (1 - pt)^gamma * log(pt)
        loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        # 5. 计算平均值
        if self.ignore_index is not None:
            active_elements_mask = labels != self.ignore_index
            return loss.sum() / active_elements_mask.sum()
        else:
            return loss.mean()

args = Seq2SeqTrainingArguments(
    # seed=Config.SEED,
    # data_seed=Config.SEED,
    num_train_epochs=Config.EPOCHS,
    learning_rate=Config.LEARNING_RATE,
    lr_scheduler_type="polynomial",
    lr_scheduler_kwargs={"lr_end": Config.MIN_LEARNING_RATE, "power": 1.0},
    weight_decay=0.01,
    # optim="adafactor",  # adamw_torch
    # label_smoothing_factor=0.1,
    
    # === Key fixes ===
    fp16=False,  
    tf32=True,
    group_by_length=False,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.INFER_BATCH_SIZE,
    gradient_checkpointing=True,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS, 
    generation_max_length=Config.INFER_MAX_NEW_TOKENS, 
    generation_num_beams=Config.INFER_NUM_BEAMS,  
    # ======================

    output_dir=Config.OUTPUT_DIR,
    eval_strategy="epoch" if Config.VALID_DS_PATH else "no",
    save_strategy="epoch",
    # save_total_limit=8,
    save_only_model=True,
    predict_with_generate=True,
    logging_steps=10,  
    report_to="swanlab",
    run_name=Config.RUN_NAME, 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid if Config.VALID_DS_PATH else None,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    # compute_loss_func=FocalLossCallback(gamma=1.0, alpha=1.0, ignore_index=-100)
)

print("Starting Training (FP32 mode)...")
trainer.train()

# ==========================================
# 4. Model inference
# ==========================================
if Config.DO_INFER and Config.VALID_DS_PATH:
    from torch.utils.data import Dataset, DataLoader

    class AkkadDataset(Dataset):
        def __init__(self, df):
            self.ids = df['uuid'].tolist()
            self.texts = []
            for text, lang in zip(df['transliteration'], df['target_lang_code']):
                if lang == "en":
                    self.texts.append(Config.PREFIX_EN + str(text))
                elif lang == "tr":
                    self.texts.append(Config.PREFIX_TR + str(text))
                elif lang == "de":
                    self.texts.append(Config.PREFIX_DE + str(text))
                else:
                    self.texts.append(Config.PREFIX_EN + str(text)) # 默认 fallback
            self.min_tokens = df['min_t'].tolist()
            self.max_tokens = df['max_t'].tolist()

        def __len__(self): return len(self.ids)

        def __getitem__(self, i):
            return {
                "id": self.ids[i],
                "text": self.texts[i],
                "min_t": self.min_tokens[i],
                "max_t": self.max_tokens[i]
            }

    def collate_fn(batch):
        ids = [x["id"] for x in batch]
        texts = [x["text"] for x in batch]
        batch_min = int(np.min([x["min_t"] for x in batch]))
        batch_max = int(np.max([x["max_t"] for x in batch]))
        
        inputs = tokenizer(texts, max_length=Config.INFER_MAX_LENGTH, padding=True, truncation=True, return_tensors="pt")
        
        return ids, inputs, batch_min, batch_max

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForSeq2SeqLM.from_pretrained(dynamic_ckpt_path, dtype=torch.float32).to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(dynamic_ckpt_path)

    loader = DataLoader(
        AkkadDataset(valid_df), 
        batch_size=Config.INFER_BATCH_SIZE, 
        shuffle=False, 
        collate_fn=collate_fn
    )

    results = []

    with torch.inference_mode():
        for ids, inputs, min_t, max_t in tqdm(loader, desc="Inference"):
            
            generate_kwargs = {
                "num_beams": Config.INFER_NUM_BEAMS,
                "do_sample": False,
                "early_stopping": True,
                "max_new_tokens": Config.INFER_MAX_NEW_TOKENS, # max_t,
                # "min_new_tokens": min_t,
                "length_penalty": 1.05,
                # "repetition_penalty": 1.0
            }

            outputs = model.generate(
                input_ids=inputs.input_ids.to(device),
                attention_mask=inputs.attention_mask.to(device),
                **generate_kwargs
            )
            
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            # cleaned = [hnr(txt) for txt in decoded]
            
            results.extend(zip(ids, decoded))

    submission = pd.DataFrame(results, columns=['uuid', 'translation'])
    submission = submission.sort_values(by='uuid', ascending=True).reset_index(drop=True)
    submission.to_csv(os.path.join(Config.OUTPUT_DIR, 'submission-train.csv'), index=False, encoding='utf-8')

Writing train-byt5.py
